In [ ]:
# 1. Connect to DuckDB and read the SQL outputs
import duckdb
import pandas as pd

con = duckdb.connect('supply_chain.duckdb')

# Read pre-built tables from SQL layer
shipping_kpi  = con.query("SELECT * FROM shipping_kpi").to_df()
category_kpi  = con.query("SELECT * FROM category_kpi").to_df()
market_kpi    = con.query("SELECT * FROM market_kpi").to_df()
fraud_kpi     = con.query("SELECT * FROM fraud_kpi").to_df()



In [ ]:
# 2. Shipping Mode scoring — MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# 2.1. Normalised shipping recommendation score
scaler = MinMaxScaler()
score_cols = ['late_delivery_pct', 'avg_actual_days', 'avg_profit_margin_pct']
shipping_score = shipping_kpi.copy()
shipping_score[['n_late', 'n_delay', 'n_margin']] = scaler.fit_transform(
    shipping_score[score_cols]
)
# Lower late % and lower delay = better; higher margin = better so we penalise the first two and reward the third => The final score is the lower the better
shipping_score['composite_score'] = (
    shipping_score['n_late']  * 0.45 +
    shipping_score['n_delay'] * 0.35 -
    shipping_score['n_margin']* 0.20
)
shipping_score['recommended'] = (
    shipping_score['composite_score'] == shipping_score['composite_score'].min()
)
shipping_score.drop(columns=['n_late', 'n_delay', 'n_margin'], inplace=True)

# 2.2. Z-score outlier flag per category
category_kpi['z_late'] = (
    (category_kpi['late_delivery_pct'] - category_kpi['late_delivery_pct'].mean())
    / category_kpi['late_delivery_pct'].std()
)
category_kpi['is_outlier'] = (category_kpi['z_late'].abs() > 1.5).astype(int)


In [ ]:
# 3. Export to CSV for Power BI
import os
os.makedirs('exports', exist_ok=True)
shipping_score.to_csv('exports/shipping_kpi.csv', index=False)
category_kpi.to_csv('exports/category_kpi.csv', index=False)
market_kpi.to_csv('exports/market_kpi.csv', index=False)
fraud_kpi.to_csv('exports/fraud_kpi.csv', index=False)
